In [1]:
import pandas as pd

df = pd.read_csv('data/thai_wikipron_5-4-2026.csv')
df

,writing,phonetic,count
0,ก,kɔː˧,2
1,ก,kɔː˧.kaj˨˩,2
2,ก.,kɔː˧,1
3,ก.ค.,kɔː˧.kʰɔː˧,1
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1
...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2
18307,ไฮ้,haj˦˥,1
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1


In [2]:
import re

def normalize_phonetic(phonetic: str) -> str:
    phonetic = (
        phonetic
        .replace('p̚', 'p')
        .replace('t̚', 't')
        .replace('k̚', 'k')
        .replace('a̯', 'ə')
        .replace('˥˩', '˦˩')
        .replace('˩˩˦', '˨˥')
    )

    SHORT_VOWELS = ['a', 'i', 'ɯ', 'u', 'e', 'ɤ', 'o', 'ɛ', 'ɔ']

    pattern = (
        '(' + '|'.join(map(re.escape, SHORT_VOWELS)) + ')'
        r'([˥˦˧˨˩]+)(?=\.|$)'
    )

    phonetic = re.sub(pattern, r'\1ʔ\2', phonetic)

    phonetic = re.sub(r'[.…]+', '.', phonetic)

    phonetic = re.sub(r'^\.|\.$', '', phonetic)

    return phonetic

print(normalize_phonetic('….daj˧.….nɯŋ˨˩'))

daj˧.nɯŋ˨˩


In [3]:
df['normalized_phonetic'] = df['phonetic'].apply(normalize_phonetic)
df

,writing,phonetic,count,normalized_phonetic
0,ก,kɔː˧,2,kɔː˧
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩
2,ก.,kɔː˧,1,kɔː˧
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧
...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩


In [4]:
import thai_gpa

def evaluate(text: str, ipa: str) -> tuple:
    result = thai_gpa.align(text, ipa)
    reconstructed_text = ''.join(s.reconstruct_text() for s in result)
    reconstructed_ipa = '.'.join(s.get_ipa(is_reduplicated=s.is_reduplicated) for s in result)
    return reconstructed_text, reconstructed_ipa

print(evaluate('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩'))

('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩')


In [5]:
from tqdm.auto import tqdm
tqdm.pandas()

def apply_evaluate(row):
    try:
        reconstructed_text, phonetic_answer = evaluate(row['writing'], row['normalized_phonetic'])
    except Exception as e:
        reconstructed_text, phonetic_answer = None, f'ERROR: {e}'
    return pd.Series([reconstructed_text, phonetic_answer])

df[['reconstructed_text', 'phonetic_answer']] = df.progress_apply(apply_evaluate, axis=1)
df.to_csv('data/test.csv', index=False)
df

C:\Users\pawi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 91%|█████████ | 16688/18310 [03:59<00:14, 111.21it/s]c:\Storage\repos\thai_grapheme_sandbox\thai_ipa.py:109: UserWarning: Warning: No explicit glottal stop at "e"
  warnings.warn(f'Warning: No explicit glottal stop at "{original}"')
100%|██████████| 18310/18310 [04:17<00:00, 71.18it/s] 


,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥,ไฮ้,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,None,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


# Analyze

In [3]:
import pandas as pd

df = pd.read_csv('data/test.csv')
df

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,NaN,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,NaN,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,NaN,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,NaN,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,NaN,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥,ไฮ้,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,NaN,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [4]:
mask = ~df["phonetic_answer"].str.startswith("ERROR:")
mismatches = df.loc[
    mask & (df["normalized_phonetic"] != df["phonetic_answer"]),
    ["writing", "normalized_phonetic", "phonetic_answer"]
]

mismatches

,writing,normalized_phonetic,phonetic_answer


In [5]:
errors = df[df["phonetic_answer"].str.startswith("ERROR:")]
errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,NaN,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,NaN,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,NaN,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,NaN,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,NaN,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18262,ไอซ์แลนด์,ʔajs˦˥.lɛːn˧,1,ʔajs˦˥.lɛːn˧,NaN,ERROR: argument of type 'NoneType' is not iter...
18263,ไอดอล,ʔaj˧.dɔl˥˩,2,ʔaj˧.dɔl˦˩,NaN,ERROR: argument of type 'NoneType' is not iter...
18278,ไอศครีม,ʔajs˧.kʰriːm˧,2,ʔajs˧.kʰriːm˧,NaN,ERROR: argument of type 'NoneType' is not iter...
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,NaN,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [6]:
errors.sample(10)

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
8167,พลาสมา,pʰlaːs˦˥.maː˥˩,2,pʰlaːs˦˥.maː˦˩,NaN,ERROR: argument of type 'NoneType' is not iter...
8820,มผ,mɔː˧.pʰɔː˩˩˦,1,mɔː˧.pʰɔː˨˥,NaN,ERROR: Could not align 'มผ' with 'mɔː˧.pʰɔː˨˥'
13304,อยู่,juː˨˩,1,juː˨˩,NaN,ERROR: Could not align 'อยู่' with 'juː˨˩'
13560,อัลตราไวโอเลต,ʔal˧.traː˥˩.waj˧.ʔoː˧.let̚˨˩,2,ʔal˧.traː˦˩.waj˧.ʔoː˧.let˨˩,NaN,ERROR: argument of type 'NoneType' is not iter...
8513,ฟอสซิล,fɔːt̚˦˥.sil˥˩,2,fɔːt˦˥.sil˦˩,NaN,ERROR: argument of type 'NoneType' is not iter...
4196,ฑ,tʰɔː˧.mon˧.tʰoː˧,2,tʰɔː˧.mon˧.tʰoː˧,NaN,ERROR: Could not align 'ฑ' with 'tʰɔː˧.mon˧.tʰ...
14805,เซลเซียส,sel˧.sia̯s˥˩,3,sel˧.siəs˦˩,NaN,ERROR: argument of type 'NoneType' is not iter...
5445,ทำนูล,tʰam˧.nuːl˧,1,tʰam˧.nuːl˧,NaN,ERROR: argument of type 'NoneType' is not iter...
16450,แคนซัส,kʰɛːn˧.sas˦˥,2,kʰɛːn˧.sas˦˥,NaN,ERROR: argument of type 'NoneType' is not iter...
15189,เบลโมแพน,bel˧.moː˧.pʰɛːn˧,1,bel˧.moː˧.pʰɛːn˧,NaN,ERROR: argument of type 'NoneType' is not iter...


In [7]:
mask = ~errors["normalized_phonetic"].str.contains(
    r"[lsf][˥˦˧˨˩]", regex=True, na=False
)
filtered_errors = errors[mask]
filtered_errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,NaN,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,NaN,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,NaN,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,NaN,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,NaN,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
17938,ไซบูทรามีน,saj˧.buː˧.tʰraː˧.miːn˧,1,saj˧.buː˧.tʰraː˧.miːn˧,NaN,ERROR: Could not align 'ไซบูทรามีน' with 'saj˧...
18041,ไปรษณียบัตร,praj˧.sa˨˩.niː˧.bat̚˨˩,2,praj˧.saʔ˨˩.niː˧.bat˨˩,NaN,ERROR: Could not align 'ไปรษณียบัตร' with 'pra...
18044,ไปรษณีย์อิเล็กทรอนิกส์,praj˧.sa˨˩.niː˧.ʔi˨˩.lek̚˦˥.tʰrɔː˧.nik̚˨˩,1,praj˧.saʔ˨˩.niː˧.ʔiʔ˨˩.lek˦˥.tʰrɔː˧.nik˨˩,NaN,ERROR: Could not align 'ไปรษณีย์อิเล็กทรอนิกส์...
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,NaN,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [43]:
filtered_errors.sample(10)

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
16232,เอนทาลปี,ʔeːn˧.tʰaw˧.piː˧,2,ʔeːn˧.tʰaw˧.piː˧,NaN,ERROR: Could not align 'เอนทาลปี' with 'ʔeːn˧....
13191,อธิบดี,ʔa˨˩.tʰi˦˥.bɔː˧.diː˧,2,ʔaʔ˨˩.tʰiʔ˦˥.bɔː˧.diː˧,NaN,ERROR: Could not align 'อธิบดี' with 'ʔaʔ˨˩.tʰ...
11392,สมมติ,som˩˩˦.mot̚˦˥,2,som˨˥.mot˦˥,NaN,ERROR: Could not align 'สมมติ' with 'som˨˥.mot˦˥'
0,ก,kɔː˧,2,kɔː˧,NaN,ERROR: Could not align 'ก' with 'kɔː˧'
17086,แอลจีเรีย,ʔɛw˧.t͡ɕiː˧.ria̯˧,3,ʔɛw˧.t͡ɕiː˧.riə˧,NaN,ERROR: Could not align 'แอลจีเรีย' with 'ʔɛw˧....
4593,ตวว,tua̯˧,1,tuə˧,NaN,ERROR: Could not align 'ตวว' with 'tuə˧'
13627,อาชกาบัต,ʔaːt͡ɕʰ˨˩.kaː˧.bat̚˨˩,1,ʔaːt͡ɕʰ˨˩.kaː˧.bat˨˩,NaN,ERROR: argument of type 'NoneType' is not iter...
8657,ภาพพจน์,pʰaːp̚˥˩.pʰot̚˦˥,1,pʰaːp˦˩.pʰot˦˥,NaN,ERROR: Could not align 'ภาพพจน์' with 'pʰaːp˦˩...
3136,ง,ŋɔː˧.ŋuː˧,2,ŋɔː˧.ŋuː˧,NaN,ERROR: Could not align 'ง' with 'ŋɔː˧.ŋuː˧'
843,การคอร์รัปชัน,kaːn˧.kʰɔː˧.rap̚˦˥.t͡ɕʰan˥˩,1,kaːn˧.kʰɔː˧.rap˦˥.t͡ɕʰan˦˩,NaN,ERROR: Could not align 'การคอร์รัปชัน' with 'k...
